In [ ]:
# ============================================================
# CATBOOST — MEDIUM DEPTH 🟡
# ============================================================
#
# No scratch implementation needed.
#
# Learn:
#   1. Why CatBoost
#   2. Core intuition
#   3. Internal working
#   4. Ordered Target Statistics
#   5. Ordered Boosting
#   6. Important parameters
#   7. sklearn-style implementation
#   8. Experiments
#   9. CatBoost vs XGBoost vs LightGBM
#   10. Interview questions
#   11. Working tree
#
# Install if needed:
# pip install catboost
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from catboost import CatBoostClassifier


# ============================================================
# 2. WHY CATBOOST?
# ============================================================
#
# CatBoost is a Gradient Boosting algorithm based on
# decision trees.
#
# Its major strength is handling CATEGORICAL FEATURES
# effectively.
#
# Example:
#
# City:
#   Mumbai
#   Pune
#   Delhi
#   Chennai
#
# Gender:
#   Male
#   Female
#
# Product:
#   Phone
#   Laptop
#   Tablet
#
# Traditional ML algorithms usually need categorical
# features converted into numbers first.
#
# CatBoost has built-in support for categorical features.
#
#
# Main ideas:
#
# Gradient Boosting
#       +
# Ordered Target Statistics
#       +
# Ordered Boosting
#       +
# Categorical Feature Handling
#
# ============================================================


# ============================================================
# 3. CREATE DATASET
# ============================================================
#
# First we'll use numerical data so that the basic CatBoost
# API is easy to understand.
#
# Later we create an actual categorical-feature example.
# ============================================================

X, y = make_classification(
    n_samples=5000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)


# ============================================================
# 4. BASIC CATBOOST MODEL
# ============================================================

# Categorical Data
#       ↓
# Ordered Target Statistics
#       ↓
# Gradient / Hessian
#       ↓
# Build Tree
#       ↓
# Correct Previous Errors
#       ↓
# Ordered Boosting
#       ↓
# More Trees
#       ↓
# Final Prediction

model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.05,
    depth=6,
    random_seed=42,
    verbose=False
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("\nAccuracy:", accuracy_score(y_test, pred))

print("\nClassification Report:")
print(classification_report(y_test, pred))


# ============================================================
# 5. CORE INTUITION
# ============================================================
#
# CatBoost is still Gradient Boosting.
#
# Basic boosting process:
#
# Initial prediction
#       ↓
# Calculate errors / gradients
#       ↓
# Build tree
#       ↓
# Add tree
#       ↓
# Calculate new errors
#       ↓
# Build next tree
#       ↓
# Repeat
#
#
# CatBoost adds special techniques to make boosting work
# better with categorical data and reduce target leakage.
#
# Two VERY IMPORTANT CatBoost concepts:
#
# 1. Ordered Target Statistics
# 2. Ordered Boosting
#
# ============================================================


# ============================================================
# 6. CATEGORICAL FEATURES
# ============================================================
#
# Suppose we have:
#
# City       Purchased
# --------------------
# Mumbai       1
# Mumbai       0
# Pune         1
# Delhi        0
# Pune         1
#
#
# We want to extract useful information from "City".
#
# A simple target encoding might calculate:
#
# Mumbai -> average target
# Pune   -> average target
# Delhi  -> average target
#
# But there is a problem:
#
# If we calculate the encoding using the current sample's
# own target value, the feature can indirectly see the answer.
#
# This is called TARGET LEAKAGE.
#
# CatBoost addresses this using:
#
# ORDERED TARGET STATISTICS
#
# ============================================================


# ============================================================
# 7. ORDERED TARGET STATISTICS ⭐⭐⭐
# ============================================================
#
# Imagine the data is randomly ordered:
#
# Row 1 -> Mumbai -> y=1
# Row 2 -> Pune   -> y=0
# Row 3 -> Mumbai -> y=1
# Row 4 -> Pune   -> y=1
#
#
# For Row 3 (Mumbai), CatBoost does NOT simply use:
#
# average of ALL Mumbai targets
#
# Instead, it uses information from PREVIOUS rows.
#
#
# Conceptually:
#
# Row 1:
#   No previous Mumbai examples
#
# Row 2:
#   No previous Pune examples
#
# Row 3:
#   Use previous Mumbai target
#
# Row 4:
#   Use previous Pune target
#
#
# Therefore:
#
# Current row
#      ↓
# Look only at previous examples
#      ↓
# Calculate category statistics
#      ↓
# Use that value as encoded feature
#
#
# This reduces target leakage.
#
# ============================================================


# ============================================================
# 8. SIMPLIFIED TARGET STATISTIC IDEA
# ============================================================
#
# A simplified form is:
#
#              previous target sum + prior
# statistic = ---------------------------
#              previous count + prior
#
#
# Example:
#
# Suppose previous "Pune" examples are:
#
# y = [1, 1, 0]
#
# Sum = 2
# Count = 3
#
# Prior = 0.5
#
# Then:
#
# statistic =
# (2 + 0.5) / (3 + 1)
#
# = 0.625
#
#
# IMPORTANT:
#
# This is a simplified explanation.
# CatBoost's actual ordered statistics include additional
# details and smoothing.
#
# For interview purposes, remember:
#
# Previous examples
#       ↓
# Category statistics
#       ↓
# Current row encoding
#
# ============================================================


# ============================================================
# 9. ORDERED BOOSTING ⭐⭐⭐
# ============================================================
#
# Standard boosting can potentially use information from the
# same training examples when calculating residuals and then
# training the next model.
#
# CatBoost uses ORDERED BOOSTING to reduce this type of
# prediction shift / target leakage.
#
#
# Conceptually:
#
# Random permutation of training data
#             ↓
# For each example
#             ↓
# Use only information available BEFORE that example
#             ↓
# Calculate training information
#             ↓
# Build boosting model
#
#
# This helps make the training process closer to how the
# model will behave on unseen data.
#
# ============================================================


# ============================================================
# 10. CREATE DATASET WITH CATEGORICAL FEATURES
# ============================================================

data = np.array([
    ["Mumbai", "Male",   "Phone"],
    ["Pune",   "Female", "Laptop"],
    ["Delhi",  "Male",   "Phone"],
    ["Mumbai", "Female", "Tablet"],
    ["Pune",   "Male",   "Phone"],
    ["Delhi",  "Female", "Laptop"],
    ["Mumbai", "Male",   "Laptop"],
    ["Pune",   "Female", "Tablet"],
    ["Delhi",  "Male",   "Tablet"],
    ["Mumbai", "Female", "Phone"],
    ["Pune",   "Male",   "Laptop"],
    ["Delhi",  "Female", "Phone"],
    ["Mumbai", "Male",   "Tablet"],
    ["Pune",   "Female", "Phone"],
    ["Delhi",  "Male",   "Laptop"],
    ["Mumbai", "Female", "Laptop"],
    ["Pune",   "Male",   "Tablet"],
    ["Delhi",  "Female", "Phone"],
    ["Mumbai", "Male",   "Phone"],
    ["Pune",   "Female", "Laptop"]
])

y_cat = np.array([
    1, 0, 0, 1, 1,
    0, 1, 0, 1, 1,
    1, 0, 1, 0, 0,
    1, 1, 0, 1, 0
])

X_cat_train, X_cat_test, y_cat_train, y_cat_test = train_test_split(
    data,
    y_cat,
    test_size=0.2,
    random_state=42
)

print("Categorical training data:")
print(X_cat_train[:5])


# ============================================================
# 11. CATBOOST WITH CATEGORICAL FEATURES
# ============================================================
#
# cat_features tells CatBoost which columns are categorical.
#
# Here:
#
# Column 0 -> City
# Column 1 -> Gender
# Column 2 -> Product
# ============================================================

cat_features = [0, 1, 2]

cat_model = CatBoostClassifier(
    iterations=100,
    learning_rate=0.05,
    depth=5,
    random_seed=42,
    verbose=False
)

cat_model.fit(
    X_cat_train,
    y_cat_train,
    cat_features=cat_features
)

cat_pred = cat_model.predict(X_cat_test)

print(
    "\nCategorical Data Accuracy:",
    accuracy_score(y_cat_test, cat_pred)
)


# ============================================================
# 12. IMPORTANT PARAMETERS
# ============================================================

# ------------------------------------------------------------
# iterations
# ------------------------------------------------------------
#
# Number of boosting iterations / trees.
#
# Similar idea to:
#
# XGBoost -> n_estimators
# LightGBM -> n_estimators
#
# Higher:
#   More model capacity
#   More training time
#   Possible overfitting
#
#
# ------------------------------------------------------------
# learning_rate
# ------------------------------------------------------------
#
# Contribution of each tree.
#
# Smaller:
#   Smaller updates
#   Usually need more iterations
#
#
# ------------------------------------------------------------
# depth ⭐
# ------------------------------------------------------------
#
# Maximum depth of individual trees.
#
# Higher:
#   More complex trees
#   Can model more complex relationships
#   Higher overfitting risk
#
#
# ------------------------------------------------------------
# l2_leaf_reg
# ------------------------------------------------------------
#
# L2 regularization applied to leaf values.
#
# Higher:
#   Stronger regularization
#   Can reduce overfitting
#
#
# ------------------------------------------------------------
# loss_function
# ------------------------------------------------------------
#
# Specifies the training loss.
#
# Examples:
#
# "Logloss"
# "CrossEntropy"
# "RMSE"
#
#
# ------------------------------------------------------------
# random_seed
# ------------------------------------------------------------
#
# Controls randomness.
#
#
# ------------------------------------------------------------
# cat_features ⭐⭐⭐
# ------------------------------------------------------------
#
# Specifies which columns are categorical.
#
# Example:
#
# cat_features=[0, 2, 5]
#
#
# ------------------------------------------------------------
# verbose
# ------------------------------------------------------------
#
# Controls training output.
#
# verbose=False
# means don't print every training iteration.
#
#
# ------------------------------------------------------------
# eval_metric
# ------------------------------------------------------------
#
# Metric used for evaluation.
#
# Example:
#
# eval_metric="Accuracy"
#
# ============================================================


# ============================================================
# 13. MODEL WITH IMPORTANT PARAMETERS
# ============================================================

model = CatBoostClassifier(
    iterations=200,
    learning_rate=0.05,
    depth=6,

    # L2 regularization
    l2_leaf_reg=3.0,

    # Evaluation metric
    eval_metric="Accuracy",

    random_seed=42,

    # Don't print training progress
    verbose=False
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print(
    "\nAccuracy:",
    accuracy_score(y_test, pred)
)


# ============================================================
# 14. EXPERIMENT — TREE DEPTH
# ============================================================
#
# Let's see how tree depth affects performance.
# ============================================================

depth_values = [2, 4, 6, 8, 10]

train_scores = []
test_scores = []

for depth in depth_values:

    model = CatBoostClassifier(
        iterations=100,
        learning_rate=0.05,
        depth=depth,
        random_seed=42,
        verbose=False
    )

    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_scores.append(
        accuracy_score(y_train, train_pred)
    )

    test_scores.append(
        accuracy_score(y_test, test_pred)
    )


plt.figure(figsize=(8, 5))

plt.plot(
    depth_values,
    train_scores,
    marker="o",
    label="Train"
)

plt.plot(
    depth_values,
    test_scores,
    marker="o",
    label="Test"
)

plt.xlabel("Tree Depth")
plt.ylabel("Accuracy")
plt.title("Effect of CatBoost Tree Depth")

plt.legend()
plt.show()


# ============================================================
# 15. FEATURE IMPORTANCE
# ============================================================

importance = model.feature_importances_

print("\nFeature Importance:")

for i, value in enumerate(importance):
    print(f"Feature {i}: {value}")


plt.figure(figsize=(8, 5))

plt.bar(
    range(len(importance)),
    importance
)

plt.xlabel("Feature")
plt.ylabel("Importance")
plt.title("CatBoost Feature Importance")

plt.show()


# ============================================================
# 16. CATBOOST vs XGBOOST vs LIGHTGBM
# ============================================================
#
# All three are gradient boosting tree algorithms.
#
#
# XGBoost:
#
# Gradient + Hessian
#       ↓
# Regularized tree boosting
#       ↓
# Strong general-purpose boosting
#
#
# LightGBM:
#
# Gradient + Hessian
#       ↓
# Histogram-based splitting
#       ↓
# Leaf-wise growth
#       ↓
# Very efficient on large tabular datasets
#
#
# CatBoost:
#
# Gradient Boosting
#       ↓
# Ordered Target Statistics
#       ↓
# Ordered Boosting
#       ↓
# Strong categorical feature handling
#
# ============================================================


# ============================================================
# 17. QUICK COMPARISON
# ============================================================
#
#                XGBoost      LightGBM       CatBoost
#
# Boosting         Yes           Yes            Yes
#
# Trees            Yes           Yes            Yes
#
# Histogram        Yes           Yes            Yes
#
# Leaf-wise        No*           Yes            Special
#
# Categorical      Limited       Limited        Excellent
# handling
#
# Ordered          No            No             Yes
# boosting
#
# Main strength:
#
# XGBoost  -> Powerful + regularized boosting
#
# LightGBM -> Speed + scalability
#
# CatBoost -> Categorical features + reduced leakage
#
# ============================================================


# ============================================================
# 18. WHEN SHOULD YOU USE CATBOOST?
# ============================================================
#
# Particularly useful when your dataset contains many
# categorical features.
#
# Examples:
#
# Customer data:
#
# City
# Country
# Product
# Device
# Browser
# Subscription type
#
#
# E-commerce:
#
# Product category
# Brand
# Seller
# Region
#
#
# CatBoost can directly handle these categorical columns.
#
# ============================================================


# ============================================================
# 19. WHEN NOT TO USE CATBOOST?
# ============================================================
#
# For purely numerical tabular data, XGBoost / LightGBM /
# CatBoost can all be reasonable choices.
#
# For:
#
# Raw images
# Raw audio
# Large text understanding
#
# Deep learning architectures are generally more appropriate.
#
# ============================================================


# ============================================================
# 20. INTERVIEW QUESTIONS
# ============================================================
#
#
# Q1. What is CatBoost?
#
# Answer:
#
# CatBoost is a gradient boosting algorithm based on
# decision trees that is particularly designed to handle
# categorical features effectively.
#
#
# ------------------------------------------------------------
#
# Q2. What problem does CatBoost solve?
#
# Answer:
#
# Traditional tree boosting algorithms often require
# categorical features to be manually encoded.
# CatBoost provides native categorical feature handling
# using techniques such as ordered target statistics.
#
#
# ------------------------------------------------------------
#
# Q3. What are Ordered Target Statistics?
#
# Answer:
#
# CatBoost converts categorical features into numerical
# statistics using information from previous examples in
# a permutation rather than directly using the current
# example's target.
#
# This helps reduce target leakage.
#
#
# ------------------------------------------------------------
#
# Q4. What is Ordered Boosting?
#
# Answer:
#
# It is a CatBoost technique where training information for
# an example is calculated using appropriate preceding
# examples, helping reduce prediction shift and leakage.
#
#
# ------------------------------------------------------------
#
# Q5. Why is CatBoost good for categorical data?
#
# Answer:
#
# It has native categorical feature handling and uses
# ordered statistics rather than requiring simple
# one-hot encoding for every category.
#
#
# ------------------------------------------------------------
#
# Q6. What is the difference between CatBoost and XGBoost?
#
# Answer:
#
# Both are gradient boosting algorithms, but CatBoost has
# specialized techniques for categorical features and
# ordered boosting.
#
#
# ------------------------------------------------------------
#
# Q7. What is the difference between CatBoost and LightGBM?
#
# Answer:
#
# LightGBM focuses heavily on efficient histogram-based
# boosting and leaf-wise tree growth, while CatBoost is
# particularly strong when categorical features are important.
#
#
# ============================================================


# ============================================================
# 21. 30-SECOND INTERVIEW EXPLANATION
# ============================================================
#
# CatBoost is a gradient boosting algorithm based on
# decision trees. Its main advantage is native handling of
# categorical features.
#
# It uses ordered target statistics to convert categorical
# information while reducing target leakage, and ordered
# boosting to reduce prediction shift.
#
# Like other boosting algorithms, it builds trees
# sequentially to correct previous errors.
#
# ============================================================


# ============================================================
# 22. FINAL MEMORY CHEATSHEET
# ============================================================
#
# CatBoost =
#
# Gradient Boosting
#       +
# Ordered Target Statistics
#       +
# Ordered Boosting
#       +
# Native Categorical Features
#
#
# MUST REMEMBER:
#
# 1. CatBoost = Gradient Boosting
#
# 2. Main strength = categorical features
#
# 3. Ordered Target Statistics
#    → categorical encoding
#    → reduces target leakage
#
# 4. Ordered Boosting
#    → reduces prediction shift / leakage
#
# 5. Important parameters:
#
#    iterations
#    learning_rate
#    depth
#    l2_leaf_reg
#    cat_features
#
# ============================================================


# ============================================================
# 23. WORKING TREE ⭐⭐⭐
# ============================================================
#
#
#                    CATBOOST
#                       │
#                       ▼
#             Gradient Boosting
#                       │
#             ┌─────────┴─────────┐
#             │                   │
#             ▼                   ▼
#       Numerical Features   Categorical Features
#                                   │
#                                   ▼
#                         Ordered Target Statistics
#                                   │
#                                   ▼
#                          Numerical Representation
#                                   │
#                                   ▼
#                       Calculate Gradients/Hessians
#                                   │
#                                   ▼
#                           Build Decision Tree
#                                   │
#                                   ▼
#                         Ordered Boosting Process
#                                   │
#                                   ▼
#                     Add Tree to Previous Model
#                                   │
#                                   ▼
#                              Repeat
#                                   │
#                                   ▼
#                            Final Prediction
#
#
# ============================================================
#
#
# ============================================================